# SynBioCrow 2.3 — BioPKS Bidirectional One-Step Rule Recovery\nReuses the successful PKS-only checkpoints. For a one-generation post-PKS bridge, keeps only JN1224MIN rules whose reactant templates match the saved PKS precursor and whose product templates match the desired final target.\n

In [ ]:
import os,sys,subprocess,shutil,json,zipfile,hashlib
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_biopks_bidir")
BIOPKS_ROOT=Path("/content/BioPKS-Pipeline")
VENV=Path("/content/biopks_py310_bidir")
PY=VENV/"bin/python"
GATE=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/biopks_reentry")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/biopks_bidirectional_rules")
OUT.mkdir(parents=True,exist_ok=True)

def run(cmd,*,cwd=None,timeout=1800,check=True):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if check and p.returncode:
        raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

if REPO.exists(): shutil.rmtree(REPO)
run(["git","clone","--depth","1","--branch","develop/2.3",
     "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
if BIOPKS_ROOT.exists(): shutil.rmtree(BIOPKS_ROOT)
run(["git","clone","--depth","1","https://github.com/JBEI/BioPKS-Pipeline.git",str(BIOPKS_ROOT)],timeout=600)
run([sys.executable,"-m","pip","install","-q","uv"],timeout=600)
run(["uv","python","install","3.10"],timeout=900)
if VENV.exists(): shutil.rmtree(VENV)
run(["uv","venv","--python","3.10",str(VENV)],timeout=600)
req=BIOPKS_ROOT/"requirements.txt"
patched=BIOPKS_ROOT/"requirements.synbiocrow_colab.txt"
patched.write_text("\n".join(x for x in req.read_text().splitlines() if x.strip().lower()!="umap==0.1.1")+"\n")
run(["uv","pip","install","--python",str(PY),"-r",str(patched)],timeout=1800)
run(["uv","pip","install","--python",str(PY),"setuptools<81"],timeout=600)
run(["uv","pip","install","--python",str(PY),"doranet==0.5.7a1"],timeout=600)
run(["uv","pip","install","--python",str(PY),"--no-deps","-e",str(BIOPKS_ROOT)],timeout=900)

panel=json.loads((REPO/"benchmarks/2.3/biopks_reentry_panel.json").read_text())
targets={x["target_id"]:x for x in panel["targets"]}
probe=REPO/"scripts/probe_biopks_bidirectional_rules_2_3.py"
summary={"schema":"synbiocrow.biopks_bidirectional_rules_summary.v1","targets":{}}

for tid in ("pks_naringenin","pks_pinocembrin"):
    gate=json.loads((GATE/f"{tid}.pks_only.json").read_text())
    routes=(gate.get("response") or {}).get("routes") or []
    top=max(routes,key=lambda x:float(x.get("score") or -1))
    precursor=top["predicted_product_smiles"]
    target=targets[tid]["target_smiles"]
    rec={"pks_precursor":precursor,"pks_score":top.get("score")}
    summary["targets"][tid]=rec
    cmd=[str(PY),str(probe),"--precursor",precursor,"--target",target,
         "--target-name",tid,"--base-ruleset","JN1224MIN","--output-dir",str(OUT)]
    try:
        p=run(cmd,cwd=OUT,timeout=120,check=False)
        rec["return_code"]=p.returncode
        rp=OUT/f"{tid}.bidirectional.json"
        rec["result"]=json.loads(rp.read_text()) if rp.is_file() else {"status":"NO_REPORT","stdout_tail":p.stdout[-5000:]}
    except subprocess.TimeoutExpired:
        rec["result"]={"status":"TIMEOUT","timeout_seconds":120}
    print("RESULT",tid,rec["result"])

report=OUT/"biopks_bidirectional_rules_summary.json"
report.write_text(json.dumps(summary,indent=2,sort_keys=True)+"\n")
print("\nSUMMARY")
print(json.dumps(summary,indent=2,sort_keys=True))

bundle=OUT/"SynBioCrow_2_3_BIOPKS_BIDIRECTIONAL_RULES_RESULTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*")):
        if p.is_file() and p.suffix in (".json",".tsv"):
            z.write(p,p.name)
    z.write(probe,probe.name)
print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))
